<a href="https://colab.research.google.com/github/Raymondycp/AI_Document_RAG/blob/main/%5BAI%5DLangchain_Docling_huggingface_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG with LangChain 🦜🔗

In [ ]:
# requirements for this example:
%pip install -qq docling docling-core python-dotenv langchain-text-splitters langchain-huggingface langchain-milvus

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

## Setup

### Loader and splitter

Below we set up:
- a `Loader` which will be used to create LangChain documents, and
- a splitter, which will be used to split these documents

In [ ]:
from typing import Iterator
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document as LCDocument
from docling.document_converter import DocumentConverter

class DoclingPDFLoader(BaseLoader):

    def __init__(self, file_path: str | list[str]) -> None:
        self._file_paths = file_path if isinstance(file_path, list) else [file_path]
        self._converter = DocumentConverter()

    def lazy_load(self) -> Iterator[LCDocument]:
        for source in self._file_paths:
            dl_doc = self._converter.convert(source).document
            text = dl_doc.export_to_markdown()
            yield LCDocument(page_content=text)

In [ ]:
FILE_PATH = "/content/deepseekr1.pdf"

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = DoclingPDFLoader(file_path=FILE_PATH)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

We now used the above-defined objects to get the document splits:

In [ ]:
docs = loader.load()
splits = text_splitter.split_documents(docs)

### Embeddings

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

HF_EMBED_MODEL_ID = "BAAI/bge-small-en-v1.5"
embeddings = HuggingFaceEmbeddings(model_name=HF_EMBED_MODEL_ID)

### Vector store

In [ ]:
from tempfile import TemporaryDirectory
from langchain_milvus import Milvus
import os

MILVUS_URI = os.environ.get(
    "MILVUS_URI", f"{(tmp_dir := TemporaryDirectory()).name}/milvus_demo.db"
)

vectorstore = Milvus.from_documents(
    splits,
    embeddings,
    connection_args={"uri": MILVUS_URI},
    drop_old=True,
)

In [ ]:
!pip install huggingface_hub ipywidgets
from huggingface_hub import notebook_login
notebook_login()

### LLM

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint
from google.colab import userdata
userdata.get('HF_TOKEN')
HF_TOKEN = os.environ.get("HF_TOKEN")

HF_LLM_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

llm = HuggingFaceEndpoint(
    repo_id=HF_LLM_MODEL_ID,
    task="text-generation",
    max_new_tokens=1024,
    do_sample=False,
    huggingfacehub_api_token=HF_TOKEN,
)

## RAG

In [ ]:
from typing import Iterable
from langchain_core.documents import Document as LCDocument
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs: Iterable[LCDocument]):
    return "\n\n".join(doc.page_content for doc in docs)


retriever = vectorstore.as_retriever()

prompt = PromptTemplate.from_template(

    "Context information is below.\n"
    "---------------------\n"
    "{context}\n"
    "---------------------\n"
    "Given the context information and no prior knowledge, answer the query.\n"
    "Query: {question}\n"
    "Answer:"
    )

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
result = rag_chain.invoke("What is the different between deepseek R1 and V3?")
print(result)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)


 DeepSeek-R1 and DeepSeek-V3 are two versions of a knowledge-based AI model. DeepSeek-R1 is an upgraded version that outperforms DeepSeek-V3 on many tasks, particularly in education-oriented knowledge benchmarks such as MMLU, MMLU-Pro, and GPQA Diamond. This improvement is primarily due to enhanced accuracy in STEM-related questions achieved through large-scale reinforcement learning. DeepSeek-R1 also excels on FRAMES, a long-context-dependent QA task, demonstrating its strong document analysis capabilities. On the other hand, DeepSeek-V3 performs better in tasks such as function calling, multi-turn, complex role-playing, and JSON output. It is also more adept at handling non-Chinese languages. However, it's important to note that DeepSeek-R1 is sensitive to prompts and may underperform in certain situations if not prompted correctly.
